Snowpark DataFrame examples with SQL string fix
*Co-authored with CoCo*

# Section: Snowpark DataFrames and VS Code Extension

In [ ]:
# Get the active Snowpark session (works inside Snowflake Notebooks)
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col,sum as sum_, avg,count
session = get_active_session()

In [ ]:
# Create a DataFrame reference to the menu table 
df = session.table("TASTY_BYTES_SAMPLE_DATA.raw_pos.menu")

In [ ]:
# Display the first 10 rows (default)
df.show()

In [ ]:
# .collect() pulls ALL rows into local memory as a list of Row objects
rows = df.collect()
print(f"Row count: {len(rows)}")
print(rows[0])

In [ ]:
from snowflake.snowpark.functions import col
# Filter the Dataframe to only include rows where truck brand is "Freezing Point"
df_freezing = df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
df_freezing.show()

In [ ]:
# Select only specific columns from the filtered DataFrame
df_selected = df_freezing.select(
col("MENU_ITEM_NAME"),
col("SALE_PRICE_USD")
)
df_selected.show()

In [ ]:
df_sql = session.sql(
"""
SELECT MENU_ITEM_NAME, SALE_PRICE_USD
FROM TASTY_BYTES_SAMPLE_DATA.raw_pos.menu
WHERE TRUCK_BRAND_NAME = 'Freezing Point'
"""
)
df_sql.show()

In [ ]:
# Create a DataFrame from local Python data (useful for small test datasets)
data = [("lemonade",4.50),("ice cream sandwich", 6.00), ("bottled water",2.00)]
df_local = session.create_dataframe(data,schema=["item","price"])
df_local.show()

## Lesson: Snowpark Dataframes Part 2

In [ ]:
# Aggregate: group by truck brand and compute count, average price, and total menu value
df_summary = (
df.group_by(col("TRUCK_BRAND_NAME")).agg(
count(col("MENU_ITEM_NAME")).alias("item_count"),
avg(col("SALE_PRICE_USD")).alias("avg_price"),
sum_(col("SALE_PRICE_USD")).alias("total_menu_value")
).sort(col("avg_price").desc())
)

df_summary.show()

In [ ]:
# .describe() computes summary stats (count, mean, stdev, min, max) for selected columns
df.select(col("SALE_PRICE_USD"),col("COST_OF_GOODS_USD")).describe().show()

In [ ]:
# Convert Snowpark DataFrame to a Pandas DataFrame (pulls data to local memory)
df_pandas = df_summary.to_pandas()
print(type(df_pandas))
df_pandas.head()


In [ ]:
# Create the analytics schema if it doesn't already exist
session.sql("CREATE SCHEMA IF NOT EXISTS TASTY_BYTES_SAMPLE_DATA.analytics").collect()

# Write the summary DataFrame to a Snowflake table (overwrite if it already exists)
df_summary.write.mode("overwrite").save_as_table(
"TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary"
)


In [ ]:
# Verify the table was created by reading it back
session.table("TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary").show()

## Assessment

In [ ]:
# Create a DataFrame reference to the menu table 
df_table = session.table("tasty_bytes.raw_pos.menu")
df_table.show()

In [ ]:
df_table = session.sql("SELECT * FROM TASTY_BYTES.RAW_POS.MENU LIMIT 10")

In [ ]:
df_table = session.table('TASTY_BYTES.RAW_POS.MENU')
df_table = df_table.filter(col("TRUCK_BRAND_NAME") == "The Mac Shack")
df_table.show()

In [ ]:
df_table.select(col("MENU_ITEM_NAME"),col("ITEM_CATEGORY")).show()
